In [7]:
from __future__ import annotations

import sys
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

sys.path.append(str(Path("..").resolve()))

from src.features.transformers import (
    StringCleaner,
    MissingIndicatorAdder,
    StructuralMissingImputer,
    BinaryYesNoEncoder,
    HasValueIndicatorEncoder,
    RareCategoryGrouper,
    FeatureEngineer,
)

RANDOM_STATE = 42
TARGET_COL = "Depression"
ID_COL = "id"

train: pd.DataFrame = pd.read_csv("../data/raw/train.csv")
test: pd.DataFrame = pd.read_csv("../data/raw/test.csv")

In [8]:
drop_cols = [ID_COL, "Name", "City"]

X = train.drop(columns= drop_cols + [TARGET_COL])
y = train[TARGET_COL]
X_test = test.drop(columns= drop_cols)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (112560, 16), Val: (28140, 16), Test: (93800, 16)


In [9]:
feature_pipeline = Pipeline(steps=[
    ("string_cleaner", StringCleaner()),
    ("has_value_indicator", HasValueIndicatorEncoder(
        cols=["Profession", "Degree"]
    )),
    ("missing_indicators", MissingIndicatorAdder(min_missing_frac=0.0)),
    ("structural_imputer", StructuralMissingImputer()),
    ("feature_engineer", FeatureEngineer()),
    ("rare_grouper", RareCategoryGrouper(min_count=30)),
])

X_train_pre = feature_pipeline.fit_transform(X_train)
X_val_pre   = feature_pipeline.transform(X_val)
X_test_pre  = feature_pipeline.transform(X_test)

print("Columns after feature_pipeline:", X_train_pre.shape[1])

Columns after feature_pipeline: 32


In [10]:
PROCESSED_DIR = Path("../data/interim")
PROCESSED_DIR.mkdir(exist_ok=True)

X_train_pre.to_csv(PROCESSED_DIR / "X_train_pre.csv", index=True)
X_val_pre.to_csv(PROCESSED_DIR  / "X_val_pre.csv",   index=True)
X_test_pre.to_csv(PROCESSED_DIR / "X_test_pre.csv",  index=True)

y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=True, header=True)
y_val.to_csv(PROCESSED_DIR   / "y_val.csv",   index=True, header=True)

print("Saved processed-data into data/interim/")

Saved processed-data into data/interim/


In [11]:
import numpy as np
from numpy.typing import NDArray
from sklearn.impute import SimpleImputer

numeric_cols: list[str] = X_train_pre.select_dtypes(include=np.number).columns.tolist()
categorical_cols: list[str] = X_train_pre.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")

# 1. Imputation layer: Fill missing values (remove NaN), no other transformations
imputer_ct = ColumnTransformer(
    transformers=[
        ("num_imp", SimpleImputer(strategy="median"), numeric_cols),
        ("cat_imp", SimpleImputer(strategy="constant", fill_value="Unknown"), categorical_cols),
    ],
    remainder="passthrough"
)

# After imputer_ct, the data becomes a numpy array with numeric columns first, then categorical columns.
new_cat_indices = list(range(len(numeric_cols), len(numeric_cols) + len(categorical_cols)))

# 2. Transformation layer: Run after SMOTENC (in modeling) or right after imputer_ct (in inference)
transform_ct = ColumnTransformer(
    transformers=[
        ("num_scale", StandardScaler(), list(range(len(numeric_cols)))),
        ("cat_ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), new_cat_indices),
    ],
    remainder="drop"
)

Numeric (26): ['Age', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Degree', 'Work/Study Hours', 'Financial Stress', 'Academic Pressure_is_missing', 'Work Pressure_is_missing', 'CGPA_is_missing', 'Study Satisfaction_is_missing', 'Job Satisfaction_is_missing', 'Dietary Habits_is_missing', 'Financial Stress_is_missing', 'Total Pressure', 'Max Pressure', 'Total Satisfaction', 'Min Satisfaction', 'Pressure_Satisfaction_Gap', 'High_Working_Hours_Flag', 'High_Financial_Stress_Flag', 'Is_Young_Adult', 'Suicide_Pressure_Interaction']
Categorical (6): ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']


In [12]:
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(feature_pipeline, MODEL_DIR / "feature_pipeline.joblib")
joblib.dump(imputer_ct, MODEL_DIR / "imputer_ct.joblib")
joblib.dump(transform_ct, MODEL_DIR / "transform_ct.joblib")

print("Saved pipelines into models/")

Saved pipelines into models/
